# Active Learning Step

Increases the validity of the initially tagged dataset prior to fine-tuning `microsoft/SportsBERT`.

**Input:** `data/initial_labels.jsonl` — noisy BIO-tagged sentences from `initial_generation.ipynb`  
**Output:** `data/conflict_candidates.jsonl` — sentences flagged for manual review  
**Output:** `data/gold_labels.jsonl` — manually corrected sentences for fine-tuning

## Load initial labels

Reads every sentence and its auto-generated BIO tags from the initial generation pass.

In [1]:
import json
from pathlib import Path

DATA_PATH = Path("../../data/initial_labels.jsonl")

rows = [json.loads(l) for l in open(DATA_PATH)]
print(f"Loaded {len(rows):,} sentences")

Loaded 226,385 sentences


## Conflict Detection

Flag sentences where the keyword regex fired but the NER model did not allocate any injury or status tags. These sentences indicate an injury mention is present but was missed — a false negative that decreases recall.

In [2]:
import re

INJURY_RE = re.compile(r"\b(injured|injury|hamstring|ankle|doubt|doubtful|ruled out)\b", re.I)
STATUS_RE  = re.compile(r"\b(suspended|suspension|ban|unavailable|miss)\b", re.I)

def has_keyword(sentence):
    return bool(INJURY_RE.search(sentence) or STATUS_RE.search(sentence))

def has_ner_tag(row):
    return any(t != "O" and t != "B-PLAYER" and t != "I-PLAYER" for t in row["tags"])

conflicts = [
    row for row in rows
    if has_keyword(row["sentence"]) and not has_ner_tag(row)
]

print(f"Conflict sentences: {len(conflicts):,} / {len(rows):,} ({100*len(conflicts)/len(rows):.1f}%)")

Conflict sentences: 2,959 / 226,385 (1.3%)


## Filter to player conflicts only

Only keep conflicts where a `B-PLAYER` tag is already present. If there is no player in the sentence, the injury keyword almost certainly refers to a non-footballer — not worth correcting.

In [3]:
player_conflicts = [
    row for row in conflicts
    if any("PLAYER" in t for t in row["tags"])
]

# Deduplicate by sentence text
seen = set()
player_conflicts_deduped = []
for row in player_conflicts:
    if row["sentence"] not in seen:
        seen.add(row["sentence"])
        player_conflicts_deduped.append(row)

print(f"Player conflicts (deduped): {len(player_conflicts_deduped):,}")

Player conflicts (deduped): 1,111


## Inspect conflicts

Sample a few flagged sentences to sanity-check quality before reviewing.

In [4]:
for row in player_conflicts_deduped[:5]:
    print("SENTENCE:", row["sentence"])
    print("TAGS:    ", list(zip(row["tokens"], row["tags"])))
    print()

SENTENCE: His life in football was never far from controversy, Allison becoming a regular in the tabloids because of his relationships with, among others, Christine Keeler of the Profumo scandal and two Miss United Kingdom winners.
TAGS:     [('His', 'O'), ('life', 'O'), ('in', 'O'), ('football', 'O'), ('was', 'O'), ('never', 'O'), ('far', 'O'), ('from', 'O'), ('controversy,', 'O'), ('Allison', 'B-PLAYER'), ('becoming', 'O'), ('a', 'O'), ('regular', 'O'), ('in', 'O'), ('the', 'O'), ('tabloids', 'O'), ('because', 'O'), ('of', 'O'), ('his', 'O'), ('relationships', 'O'), ('with,', 'O'), ('among', 'O'), ('others,', 'O'), ('Christine', 'O'), ('Keeler', 'O'), ('of', 'O'), ('the', 'O'), ('Profumo', 'O'), ('scandal', 'O'), ('and', 'O'), ('two', 'O'), ('Miss', 'O'), ('United', 'O'), ('Kingdom', 'O'), ('winners.', 'O')]

SENTENCE: Burkina Faso are also without their key striker Alain Traore, scorer of three goals in the group stages, but injured against Zambia before the knockout stages.
TAGS:  

## Save conflict candidates

Writes flagged sentences to `data/conflict_candidates.jsonl` for reference.

In [5]:
OUT_PATH = Path("../../data/conflict_candidates.jsonl")

with open(OUT_PATH, "w") as f:
    for row in player_conflicts_deduped:
        f.write(json.dumps(row) + "\n")

print(f"Saved {len(player_conflicts_deduped):,} conflict candidates to {OUT_PATH}")

Saved 1,111 conflict candidates to ../../data/conflict_candidates.jsonl


## Human-in-the-loop Review

Steps through each conflict one by one. For each sentence:
- Tokens are printed with their index so you can reference them by number
- Current (incorrect) tags are shown
- Commands:
  - **Enter** → skip (auto-label was correct or sentence is noise)
  - **`s`** → skip explicitly
  - **`q`** → quit and save progress
  - **`<tag> <idx> [idx2]`** → assign a tag to a token or range, e.g. `B-INJURY 5` or `B-INJURY 5 6` to tag tokens 5 and 6 as `B-INJURY I-INJURY`

Corrections are saved to `data/gold_labels.jsonl`. Re-running the cell resumes from where you left off.

In [6]:
from IPython.display import clear_output

GOLD_PATH  = Path("../../data/gold_labels.jsonl")
VALID_TAGS = {"B-PLAYER", "I-PLAYER", "B-INJURY", "I-INJURY", "B-STATUS", "I-STATUS", "O"}

# Load already-reviewed sentences so re-running resumes progress
reviewed_sentences = set()
if GOLD_PATH.exists():
    for line in open(GOLD_PATH):
        reviewed_sentences.add(json.loads(line)["sentence"])

remaining = [
    row for row in player_conflicts_deduped
    if row["sentence"] not in reviewed_sentences
]

def render(i, total, tokens, tags, message=""):
    clear_output(wait=True)
    print(f"─── {i+1}/{total} ───   (gold saved: {len(reviewed_sentences)})")
    print()
    print("SENTENCE:", " ".join(tokens))
    print()
    for idx, (tok, tag) in enumerate(zip(tokens, tags)):
        marker = "◀" if tag != "O" else " "
        print(f"  [{idx:>2}] {tok:<25} {tag} {marker}")
    print()
    if message:
        print(message)
        print()
    print("Commands: Enter/s=skip | q=quit | <TAG> <idx> [idx2]")
    print("          e.g.  B-INJURY 4   or   B-STATUS 2 3")

gold_file = open(GOLD_PATH, "a")

for i, row in enumerate(remaining):
    tokens = row["tokens"]
    tags   = list(row["tags"])
    msg    = ""

    while True:
        render(i, len(remaining), tokens, tags, msg)
        msg = ""
        cmd = input("> ").strip()

        if cmd == "" or cmd.lower() == "s":
            break

        if cmd.lower() == "q":
            gold_file.close()
            clear_output(wait=True)
            print(f"Quit. {len(reviewed_sentences)} gold labels saved to {GOLD_PATH}")
            raise SystemExit

        parts = cmd.split()
        if len(parts) < 2:
            msg = "✗ Unrecognised — try:  B-INJURY 4  or  B-STATUS 2 3"
            continue

        tag_input = parts[0].upper()
        if tag_input not in VALID_TAGS:
            msg = f"✗ Unknown tag '{tag_input}'. Valid: {', '.join(sorted(VALID_TAGS))}"
            continue

        try:
            indices = [int(p) for p in parts[1:]]
        except ValueError:
            msg = "✗ Indices must be integers"
            continue

        if any(idx < 0 or idx >= len(tokens) for idx in indices):
            msg = f"✗ Index out of range (0–{len(tokens)-1})"
            continue

        for pos, idx in enumerate(indices):
            if pos == 0:
                tags[idx] = tag_input
            else:
                tags[idx] = "I-" + tag_input[2:] if tag_input.startswith("B-") else tag_input

        msg = "✓ Tags updated — Enter to save and continue, or keep editing"

    if tags != list(row["tags"]):
        corrected = {"sentence": row["sentence"], "tokens": tokens, "tags": tags}
        gold_file.write(json.dumps(corrected) + "\n")
        gold_file.flush()
        reviewed_sentences.add(row["sentence"])

gold_file.close()
clear_output(wait=True)
print(f"Done. {len(reviewed_sentences)} gold labels total in {GOLD_PATH}")

Quit. 101 gold labels saved to ../../data/gold_labels.jsonl


SystemExit: 

/Users/alexy/CSE/Sports-NLP-Outcome-Predictor/.venv/lib/python3.14/site-packages/IPython/core/interactiveshell.py:3756: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


limitations: looking at sentence only gives if they are mentioned - not "he", and ending/ punctuation does not get caught players and injury tags inclusive. eg. hamstring tear. and `player_name'`s